<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB10_A_Complete_ML_Project_Yacht_Hull_Resistance_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB10 · Clase 10 — Un proyecto de ML completo: predecir la resistencia residual de un casco de velero**

## Bloque 2: IA — Machine Learning (cierre)

`NB07`–`NB09` se centraron cada uno en una pieza del conjunto de herramientas de Machine Learning: el flujo de trabajo básico, algoritmos concretos, aprendizaje no supervisado. Esta clase cierra el Bloque 2 juntando **todas las piezas en un proyecto completo**, de principio a fin, sobre un dataset que todavía no hemos tocado: el clásico dataset **[Yacht Hydrodynamics](https://archive.ics.uci.edu/dataset/243/yacht+hydrodynamics)** (308 experimentos reales de canal de ensayos con modelos de casco de velero, de la Universidad Técnica de Delft), ya disponible en este repositorio en [`Datasets/yacht_hydrodynamics.data`](https://github.com/JuanZapa7a/AINavalEngineering/blob/main/Datasets/yacht_hydrodynamics.data).

**El problema**: dada la geometría de un casco (5 coeficientes adimensionales) y su velocidad (a través del [número de Froude](https://en.wikipedia.org/wiki/Froude_number)), predecir su **resistencia residual** — el arrastre que sufre un casco además de la friccional, y una magnitud central en el diseño de buques.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Describir las etapas de un proyecto real de ML, desde el planteamiento del problema hasta un modelo desplegable.
- Realizar comprobaciones básicas de curación y calidad de datos antes de modelar.
- Usar correlación e importancia de características juntas para razonar sobre la selección de características.
- Construir y ajustar un `Pipeline` completo con `GridSearchCV`, evaluando un modelo final una sola vez, al final.
- Leer una curva de aprendizaje para diagnosticar si un modelo se beneficiaría de más datos o ya está cerca de su techo.
- Guardar un modelo entrenado en disco y recargarlo — el último paso antes de que un modelo pueda usarse operativamente.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso de NB01–NB09, hoja de ruta de hoy | 5 min | Teoría |
| 2 | El flujo de trabajo de un proyecto de ML: plantear un proyecto de principio a fin | 10 min | Teoría |
| 3 | Cargar y explorar el dataset real (resistencia del casco de un velero) | 15 min | Práctica |
| 4 | Curación de datos y comprobaciones de calidad | 10 min | Teoría + Práctica |
| 5 | Selección de características: correlación e importancia juntas | 15 min | Práctica |
| 6 | Construir un pipeline de comparación de modelos | 20 min | Práctica |
| 7 | Ajuste de hiperparámetros con `GridSearchCV` | 15 min | Práctica |
| 8 | Evaluación final: residuos y curva de aprendizaje | 20 min | Práctica |
| 9 | Guardar y recargar un modelo entrenado | 5 min | Práctica |
| 10 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son orientación aproximada, no un guion estricto — no hay pausas programadas. Si se cubre todo con tiempo de sobra, la clase termina antes; eso puede pasar y no pasa nada.

---

## 1. Repaso: dónde estamos

- **`NB01`–`NB02`**: historia de la IA, fundamentos de Python/Colab/NumPy/Pandas.
- **`NB07`**: el flujo de trabajo de ML supervisado, métricas, un clasificador y un regresor, fuga de datos.
- **`NB08`**: algoritmos de clasificación en profundidad — árboles, ensembles, SVM, pipelines a prueba de fugas, ajuste.
- **`NB09`**: aprendizaje no supervisado — clustering con K-Means, PCA.
- **`NB10`** (hoy): todo lo anterior, combinado en un proyecto completo sobre un dataset real totalmente nuevo.

Esto cierra el **Bloque 2 — IA: Machine Learning**. `NB11` abre el Bloque 3 (Deep Learning).

---

## 2. El flujo de trabajo de un proyecto de ML

Todo proyecto real de ML — no solo el ejemplo de juguete de hoy — sigue aproximadamente las mismas etapas:

| Etapa | Pregunta que responde | Dónde la hemos visto |
|---|---|---|
| **1. Planteamiento del problema** | ¿Qué estamos prediciendo exactamente, y por qué importa? | Hoy: predecir la resistencia residual a partir de la geometría del casco |
| **2. Recogida de datos** | ¿De dónde vienen los datos, podemos fiarnos? | `!wget` desde un dataset real, publicado y replicado en este repositorio |
| **3. Exploración (EDA)** | ¿Qué pinta tienen realmente los datos? | `NB02`–`NB09`: `head`, `describe`, `groupby`, `.corr()`, gráficos |
| **4. Curación de datos** | ¿Están suficientemente limpios para modelar? | Nuevo hoy: duplicados, rangos, valores ausentes, comprobaciones de sensatez |
| **5. Selección de características** | ¿Qué variables de entrada ayudan realmente? | Nuevo hoy: correlación + importancia basada en modelo, juntas |
| **6. Modelado** | ¿Qué algoritmo(s) se ajustan al problema? | `NB07`/`NB08`: regresión lineal, Random Forest, y compañía |
| **7. Ajuste** | ¿Podemos mejorar los valores por defecto? | `NB08` introdujo `GridSearchCV`; hoy lo usamos de verdad |
| **8. Evaluación final** | ¿Qué tan bueno es el modelo *final*, con honestidad? | El conjunto de test se toca exactamente una vez, al final |
| **9. Persistencia / despliegue** | ¿Cómo se usaría realmente este modelo? | Nuevo hoy: guardar y recargar un modelo entrenado |

Las dos etapas genuinamente nuevas hoy son la **4** (curación de datos) y la **9** (persistencia) — todo lo demás se apoya directamente en herramientas de `NB07`–`NB09`.

> **Para saber más**: [CRISP-DM, un modelo de proceso estándar de minería de datos (Wikipedia)](https://en.wikipedia.org/wiki/Cross-industry_standard_process_for_data_mining).

---

## 3. Cargar y explorar el dataset real

El dataset **[Yacht Hydrodynamics](https://archive.ics.uci.edu/dataset/243/yacht+hydrodynamics)** procede de ensayos en canal de remolque de 22 formas de casco de velero en Delft, a distintas velocidades — 308 medidas experimentales reales en total. Seis variables numéricas describen la geometría del casco y la velocidad; la séptima columna es el objetivo.

In [ ]:
!wget -q -O yacht.data https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/yacht_hydrodynamics.data

import pandas as pd

columns = [
    "LongPos_COB",       # longitudinal position of the center of buoyancy
    "Prismatic_Coeff",   # prismatic coefficient
    "LengthDisp_Ratio",  # length-displacement ratio
    "BeamDraft_Ratio",   # beam-draught ratio
    "LengthBeam_Ratio",  # length-beam ratio
    "Froude_Number",     # Froude number (speed)
    "Residuary_Resistance",  # target: residuary resistance per unit weight of displacement
]
yacht = pd.read_csv("yacht.data", sep=r"\s+", names=columns)
print(yacht.shape)
yacht.head()

Las cinco primeras columnas son fijas para cada diseño de casco (no cambian con la velocidad); `Froude_Number` varía porque cada casco se ensayó a varias velocidades — por eso 22 cascos producen 308 filas.

In [ ]:
yacht.describe()

---

## 4. Curación de datos y comprobaciones de calidad

Antes de modelar nada, unas comprobaciones rápidas y estándar — las mismas preguntas que merece cualquier dataset real, sin importar lo limpio que parezca:

In [ ]:
print("missing values per column:")
print(yacht.isna().sum())
print()
print("duplicate rows:", yacht.duplicated().sum())
print()
print("Froude number range:", yacht["Froude_Number"].min(), "-", yacht["Froude_Number"].max())
print("Residuary resistance range:", yacht["Residuary_Resistance"].min(), "-", yacht["Residuary_Resistance"].max())

**Lee tu propia salida**: un número de Froude real para un casco de desplazamiento debería ser un valor positivo pequeño (aproximadamente 0.1–0.5 aquí); la resistencia residual debería ser no negativa. Si alguno de los dos rangos pareciera físicamente imposible (resistencia negativa, números de Froude en los miles), `sería señal de un error de entrada de datos o un desajuste de unidades` — merece la pena detectarlo *antes* de que un modelo aprenda silenciosamente a ajustar un sinsentido.

> **Para saber más**: [Curación de datos (Wikipedia)](https://en.wikipedia.org/wiki/Data_curation).

---

## 5. Selección de características: correlación e importancia juntas

Con solo 6 características candidatas, no necesitamos estrictamente eliminar ninguna en este dataset — pero el *razonamiento* de la selección de características importa incluso aquí, y importará más en datasets reales más grandes. Dos vistas complementarias:

1. **Correlación con el objetivo** — una primera mirada rápida, sin modelo.
2. **Importancia de características basada en modelo** (como en `NB08`) — captura relaciones no lineales que la correlación por sí sola pasaría por alto.

In [ ]:
yacht.corr()["Residuary_Resistance"].sort_values(ascending=False)

`Froude_Number` debería destacar como, con diferencia, la correlación lineal más fuerte — tiene sentido físico, ya que la resistencia crece de forma pronunciada (no lineal) con la velocidad. Las cinco columnas de geometría del casco correlacionan mucho más débilmente *por separado* — pero eso no significa que sean inútiles: la resistencia depende de cómo **interactúan** la forma del casco y la velocidad, algo que una simple correlación por pares no puede capturar. Comprobémoslo con un modelo que *sí* pueda capturar interacciones:

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = yacht.drop(columns="Residuary_Resistance")
y = yacht["Residuary_Resistance"]

rf_importance_check = RandomForestRegressor(n_estimators=300, random_state=42)
rf_importance_check.fit(X, y)

pd.Series(rf_importance_check.feature_importances_, index=X.columns).sort_values(ascending=False)

**Compara las dos clasificaciones**: ¿sigue dominando `Froude_Number`? ¿Alguna columna de geometría del casco sube notablemente de posición aquí respecto a la correlación simple de antes — evidencia de que importa principalmente *en combinación* con la velocidad, no por sí sola? Mantendremos las 6 características de aquí en adelante, pero `este es exactamente el razonamiento que usarías para justificar eliminar una columna en un dataset más grande y más sucio`.

> **Para saber más**: [Selección de características (Wikipedia)](https://en.wikipedia.org/wiki/Feature_selection).

---

## 6. Construir un pipeline de comparación de modelos

Divide los datos una sola vez, reserva el conjunto de test para *todo el resto de este notebook*, y compara tres regresores — una línea base lineal más dos ensembles, en eco a la comparación bagging-vs-boosting de `NB08` — cada uno dentro de un `Pipeline` con escalado, para una comparación justa y a prueba de fugas mediante validación cruzada solo sobre los datos de entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape, " Test (untouched until Part 8):", X_test.shape)

Compara regresión lineal, Random Forest y Gradient Boosting — un ensemble *secuencial*, de tipo boosting, para regresión, el equivalente en regresión del AdaBoost de `NB08`:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score

candidate_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model in candidate_models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="r2")
    cv_results[name] = scores

pd.DataFrame(cv_results).mean().sort_values(ascending=False)

La tabla anterior muestra la R² media, ocultando cuán *consistente* es cada modelo entre folds. Un boxplot muestra ambas cosas a la vez:

In [ ]:
import matplotlib.pyplot as plt

pd.DataFrame(cv_results).boxplot(figsize=(8, 5))
plt.ylabel("Cross-validated R2")
plt.title("Model comparison — Yacht Hull Resistance")
plt.show()


**Interpreta tus propios resultados**: ¿alguno de los ensembles supera claramente a la regresión lineal simple? Si la relación de `Froude_Number` con la resistencia es fuertemente no lineal (como sugiere la física), `los ensembles deberían tener aquí una ventaja real` — a diferencia de un dataset más cercano a lo lineal, donde el modelo más simple puede ser igual de bueno. Llevaremos el modelo con mejor rendimiento a la Parte 7 para ajustarlo.

> **Para saber más**: [documentación de `sklearn.ensemble.GradientBoostingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html).

**Pruébalo tú mismo**: la Parte 5 argumentaba que una columna de geometría del casco que aporta poco por sí sola podría seguir importando, o podría ser genuinamente eliminable. Compruébalo directamente — elimina la columna más débil por importancia (`LengthDisp_Ratio`) y vuelve a ejecutar la validación cruzada de Random Forest. ¿Eliminarla ayuda, perjudica, o no supone una diferencia real?

In [ ]:
weakest_feature = "LengthDisp_Ratio"

pipe_full = Pipeline([("scaler", StandardScaler()), ("model", RandomForestRegressor(n_estimators=300, random_state=42))])
pipe_reduced = Pipeline([("scaler", StandardScaler()), ("model", RandomForestRegressor(n_estimators=300, random_state=42))])

cv_full = cross_val_score(pipe_full, X_train, y_train, cv=cv, scoring="r2")
cv_reduced = cross_val_score(pipe_reduced, X_train.drop(columns=[weakest_feature]), y_train, cv=cv, scoring="r2")

print(f"CV R2 with all 6 features: {cv_full.mean():.4f}")
print(f"CV R2 without '{weakest_feature}': {cv_reduced.mean():.4f}")


---

## 7. Ajuste de hiperparámetros con `GridSearchCV`

Toma el modelo más fuerte de la Parte 6 (**Gradient Boosting**, según la comparación con validación cruzada real de arriba — cambia el código de abajo si tus propios resultados favorecieron a Random Forest) y busca sobre sus hiperparámetros principales: `n_estimators` (cuántos árboles), `max_depth` (cuán profundo puede crecer cada árbol — recuerda la demostración de sobreajuste-frente-a-profundidad de `NB08`), y `min_samples_leaf` (un segundo freno, más suave, sobre la complejidad del árbol).

In [ ]:
from sklearn.model_selection import GridSearchCV

tuning_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GradientBoostingRegressor(random_state=42)),
])

param_grid = {
    "model__n_estimators": [100, 300, 500],
    "model__max_depth": [3, 5, 10],   # 3 is GradientBoostingRegressor's own default
    "model__min_samples_leaf": [1, 2, 4],
}

grid = GridSearchCV(tuning_pipe, param_grid, cv=cv, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV R2:", round(grid.best_score_, 3))

Compara `grid.best_score_` con la fila de Gradient Boosting sin ajustar de la Parte 6 — la búsqueda en rejilla incluye los valores por defecto sin ajustar como uno de sus candidatos, así que `el ajuste debería igualarlos o superarlos, nunca hacerlo peor`.

**Pruébalo tú mismo**: `grid.best_params_` solo muestra la única configuración ganadora. Mira en su lugar las 5 mejores de `grid.cv_results_` — ¿hay otras configuraciones casi igual de buenas, quizá más simples (menos árboles, menos profundidad) o más estables (`std_test_score` más bajo)?

In [ ]:
results_grid = pd.DataFrame(grid.cv_results_)
top5 = results_grid.sort_values("mean_test_score", ascending=False).head(5)
top5[["param_model__n_estimators", "param_model__max_depth", "param_model__min_samples_leaf", "mean_test_score", "std_test_score"]]


---

## 8. Evaluación final: residuos y curva de aprendizaje

Ahora — y solo ahora — tocamos el conjunto de test que reservamos en la Parte 6.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test, y_pred), 3))
print("Test RMSE:", round(mean_squared_error(y_test, y_pred) ** 0.5, 3))
print("Test R2:", round(r2_score(y_test, y_pred), 3))

Antes de mirar los residuos, la vista clásica de predicho-frente-a-real: los puntos cerca de la diagonal son cascos que el modelo predijo bien.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
lims = [y.min(), y.max()]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual residuary resistance")
plt.ylabel("Predicted residuary resistance")
plt.title("Final tuned model — predicted vs. actual")
plt.legend()
plt.show()


Compara esta R² de test con la R² de validación cruzada de las Partes 6–7 — deberían estar cerca. Una puntuación de test mucho peor que la de validación cruzada sugeriría que el propio proceso de ajuste sobreajustó a los datos de entrenamiento (un riesgo de fuga más sutil, ya que `GridSearchCV` prueba muchas configuraciones y podría tener suerte con alguna por azar).

Un gráfico de residuos muestra *dónde* el modelo tiene dificultades, no solo un error medio:

In [ ]:
import matplotlib.pyplot as plt

residuals = y_test - y_pred

plt.figure(figsize=(6, 5))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted residuary resistance")
plt.ylabel("Residual (actual − predicted)")
plt.title("Residual plot — final tuned model")
plt.show()

Unos residuos repartidos de forma uniforme alrededor de cero, sin un patrón evidente ni forma de embudo, sugieren que los errores del modelo son razonablemente consistentes en todo el rango de predicción. Una forma de embudo (errores que crecen con la resistencia predicha) sugeriría que `el modelo es menos fiable para cascos de alta resistencia` — algo realmente útil de saber antes de confiar en él de forma operativa.

Un diagnóstico más: una **curva de aprendizaje** muestra cómo cambia el rendimiento a medida que le damos al modelo más datos de entrenamiento — nos dice si recoger más datos siquiera ayudaría.

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train, cv=cv, scoring="r2",
    train_sizes=np.linspace(0.2, 1.0, 6), random_state=42,
)

plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training R2")
plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="Validation R2")
plt.xlabel("Training set size")
plt.ylabel("R2 score")
plt.title("Learning curve")
plt.legend()
plt.show()

**Lee tu propia curva**: si la curva de validación sigue subiendo y no se ha encontrado con la curva de entrenamiento en el borde derecho del gráfico, más datos probablemente ayudarían. Si ambas curvas se han aplanado y convergido, este modelo ha alcanzado más o menos su techo con este conjunto de características — más *datos* no ayudarían mucho, pero mejores *características* o un *algoritmo* distinto sí podrían.

> **Para saber más**: [documentación de `sklearn.model_selection.learning_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html).

---

## 9. Guardar y recargar un modelo entrenado

`Entrenar un modelo no sirve de nada si solo existe dentro de una sesión de notebook`. `joblib` serializa un objeto de scikit-learn ya entrenado (incluyendo un `Pipeline` completo) en un fichero, para poder recargarlo — en otro notebook, un script, o un pequeño servicio web — sin volver a entrenarlo.

In [ ]:
import joblib

joblib.dump(best_model, "yacht_resistance_model.pkl")
print("Model saved.")

reloaded_model = joblib.load("yacht_resistance_model.pkl")
print("Reloaded model predicts:", reloaded_model.predict(X_test.iloc[[0]]))
print("Actual value was:       ", y_test.iloc[0])

Ese es el recorrido completo de un proyecto real de ML: datos en crudo por un lado, un modelo reutilizable, probado y guardado por el otro.

> **Para saber más**: [guía de scikit-learn sobre persistencia de modelos](https://scikit-learn.org/stable/model_persistence.html) · [documentación de `joblib`](https://joblib.readthedocs.io/en/stable/).

---

## Resumen de la clase

- Un proyecto real de ML sigue un conjunto de etapas bastante consistente: plantear el problema, obtener y curar los datos, explorarlos, seleccionar características, modelar, ajustar, evaluar una sola vez al final, y dejar el resultado persistido.
- La curación de datos (valores ausentes, duplicados, rangos físicamente sensatos) es una comprobación rápida pero esencial antes de modelar — sobre todo con datos que no has recogido tú mismo.
- La correlación y la importancia de características basada en modelo responden preguntas distintas; usar ambas da una imagen más completa que cualquiera de las dos por separado, especialmente cuando las características interactúan (forma del casco × velocidad, aquí).
- `GridSearchCV` dentro de un `Pipeline` ajusta hiperparámetros sin filtrar datos de test al proceso.
- El conjunto de test se toca exactamente una vez, al final — y su puntuación debería coincidir aproximadamente con la de validación cruzada, o algo ha ido mal antes.
- Una curva de aprendizaje te dice si más datos ayudarían; un gráfico de residuos te dice dónde se concentran los errores de un modelo.
- `joblib` convierte un modelo entrenado de un artefacto solo-de-notebook en algo reutilizable en otro sitio.

## Para la próxima clase (NB11)

Abrimos el **Bloque 3 — Deep Learning**: redes neuronales, desde un único perceptrón hasta los bloques de construcción de las CNN, cubriendo el mayor hueco de contenidos identificado frente a la guía docente del curso.

## Tarea / Ideas de práctica

1. Repite la comparación de modelos de la Parte 6, pero añade `SVR` (Support Vector Regression, el equivalente en regresión del `SVC` de `NB08`) a los candidatos — ¿cómo se compara con los ensembles?
2. Amplía la búsqueda en rejilla de la Parte 7: añade `model__max_features` como cuarto hiperparámetro ajustado — ¿cambia la mejor configuración encontrada?
3. En la Parte 8, identifica los 5 cascos del conjunto de test con los residuos más grandes (en valor absoluto) — ¿tienen algo en común (un rango concreto de número de Froude, o una geometría de casco)?
4. Repite la curva de aprendizaje de la Parte 8 usando solo `Froude_Number` como característica (eliminando las 5 columnas de geometría del casco) — ¿cuánto empeora el techo, y qué te dice eso sobre cuánto están aportando realmente esas 5 columnas?
5. Carga tu `yacht_resistance_model.pkl` guardado en un runtime de Colab *nuevo* (Entorno de ejecución → Reiniciar sesión) y confirma que sigue prediciendo correctamente sin volver a ejecutar las Partes 1–8 — esta es la prueba real de si la persistencia funcionó de verdad.

> ***Como siempre: un modelo ajustado solo es tan fiable como la curación de datos y la evaluación final honesta que hay detrás — una R² alta en un conjunto de test con fugas es peor que una R² modesta en uno limpio.***